In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

### ***-: UPLOADING .csv AND CONVERTING AS DATAFRAME :-***

In [ ]:
from google.colab import files
import io

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

print("DataFrame created successfully:")
print(df.head())

### ***-: BASIC DATA QUALITY CHECKS :-***

In [ ]:
df.info()
df.head(10)
df.describe()

### ***-: TEXT CLEANING & PREPROCESSING :-***

In [ ]:
#1. Basic Text Normalization :-

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)              # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning
df["clean_text"] = df["feedback_text"].apply(clean_text)

df[["feedback_text", "clean_text"]].head()

In [ ]:
#2. Stopword Removal :-

stopwords = ENGLISH_STOP_WORDS

def remove_stopwords(text):
    return " ".join(
        word for word in text.split()
        if word not in stopwords
    )

df["clean_text"] = df["clean_text"].apply(remove_stopwords)

df[["feedback_text", "clean_text"]].head()

In [ ]:
#3. Handle Empty / Short Feedback :-

df["text_length"] = df["clean_text"].str.split().apply(len)

df["text_length"].describe()


In [ ]:
#4. Filter Extremely Short Feedback :-

df = df[df["text_length"] >= 2].reset_index(drop=True)

In [ ]:
#5. Sanity Check After Cleaning :-

df.sample(10)[["feedback_text", "clean_text", "rating"]]
df.isnull().sum()

### ***-: TEXT CLEANING SUMMARY :-***

**Text Cleaning & Preprocessing**

Customer feedback text was normalized by lowercasing, removing punctuation,
numbers, and generic stopwords while preserving domain-specific terms.
Extremely short or non-informative feedback was filtered to ensure meaningful
analysis.

The preprocessing approach prioritizes interpretability and business context
over aggressive linguistic transformations.

### ***-: EXPLORATORY TEXT ANALYSIS :-***

In [ ]:
#1. Most Common Words :-

from collections import Counter

all_words = " ".join(df["clean_text"]).split()
word_freq = Counter(all_words)

word_freq.most_common(20)

In [ ]:
#2. Visualize Top Words :-

top_words = word_freq.most_common(15)

words, counts = zip(*top_words)

plt.figure(figsize=(8,4))
sns.barplot(x=list(counts), y=list(words))
plt.title("Most Frequent Words in Customer Feedback")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.show()

In [ ]:
#3. Negative Feedback Language :-

negative_text = df[df["rating"] <= 2]["clean_text"]
neg_words = Counter(" ".join(negative_text).split())

neg_words.most_common(15)

In [ ]:
#4. Positive Feedback Language :-

positive_text = df[df["rating"] >= 4]["clean_text"]
pos_words = Counter(" ".join(positive_text).split())

pos_words.most_common(15)

In [ ]:
#5. Compare Complaint Volume by Channel :-

df.groupby("channel")["feedback_id"].count().plot(
    kind="bar",
    figsize=(6,4),
    title="Feedback Volume by Channel"
)
plt.ylabel("Number of Feedbacks")
plt.show()

In [ ]:
#6. Keyword-Based Issue Buckets :-

issue_keywords = {
    "payment": ["payment", "refund", "deducted"],
    "app_performance": ["crash", "slow", "buggy", "loading"],
    "order_delivery": ["delivery", "order", "cancelled", "track"],
    "support": ["support", "service"]
}

def detect_issue(text):
    for issue, keywords in issue_keywords.items():
        if any(word in text for word in keywords):
            return issue
    return "other"

df["issue_bucket"] = df["clean_text"].apply(detect_issue)

df["issue_bucket"].value_counts()

### ***-: TEXT ANALYSIS SUMMARY :-***

**Exploratory Text Analysis**

Initial text exploration revealed recurring customer pain points related to payments, refunds, app performance, order management, and customer support. Negative feedback contained more specific and actionable language, while positive feedback tended to be generic.

Simple keyword-based grouping already highlighted dominant issue categories, indicating that customer complaints are highly concentrated rather than random.

### ***-: TOPIC MODELING [LDA] :-***

In [ ]:
#1. Vectorize Text :-

vectorizer = CountVectorizer(
    max_df=0.85,
    min_df=10,
    stop_words="english"
)

dtm = vectorizer.fit_transform(df["clean_text"])

dtm.shape

In [ ]:
#2. Train LDA Model :-

lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="batch"
)

lda_model.fit(dtm)

In [ ]:
#3. Inspect Topics :-

def display_topics(model, feature_names, n_top_words=8):
    for idx, topic in enumerate(model.components_):
        print(f"\nTopic {idx+1}:")
        print(
            ", ".join(
                [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
            )
        )

display_topics(lda_model, vectorizer.get_feature_names_out())

In [ ]:
#4. Assign Dominant Topic to Each Feedback :-

topic_probs = lda_model.transform(dtm)
df["dominant_topic"] = topic_probs.argmax(axis=1)

df[["feedback_text", "dominant_topic"]].head()

In [ ]:
#5. Topic Distribution :-

topic_counts = df["dominant_topic"].value_counts().sort_index()

plt.figure(figsize=(7,4))
sns.barplot(
    x=topic_counts.index,
    y=topic_counts.values
)
plt.title("Customer Feedback Distribution by Topic")
plt.xlabel("Topic")
plt.ylabel("Number of Feedbacks")
plt.show()

In [ ]:
#6. Topic vs Rating :-

df.groupby("dominant_topic")["rating"].mean()

### ***-: TOPIC MODELING SUMMARY :-***

**Topic Modeling (Business Themes)**

Latent Dirichlet Allocation (LDA) was applied to customer feedback to uncover key issue themes without predefined labels. The model identified distinct topics related to payments, app performance, order management, customer support, and general experience.

Topic distribution and average ratings revealed that a small number of themes account for the majority of negative feedback, enabling targeted prioritization of improvement efforts.

### ***-: SENTIMENT & THEME IMPACT ANALYSIS :-***

In [ ]:
#1. Create a Simple Sentiment Score :-

def sentiment_label(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df["sentiment"] = df["rating"].apply(sentiment_label)

df["sentiment"].value_counts()

In [ ]:
#2. Sentiment Distribution by Topic :-

topic_sentiment = (
    df.groupby(["dominant_topic", "sentiment"])
      .size()
      .unstack(fill_value=0)
)

topic_sentiment

In [ ]:
#3. Visualize Topic × Sentiment Heatmap :-

plt.figure(figsize=(8,5))
sns.heatmap(
    topic_sentiment,
    annot=True,
    fmt="d",
    cmap="Reds"
)
plt.title("Sentiment Distribution Across Topics")
plt.xlabel("Sentiment")
plt.ylabel("Topic")
plt.show()

In [ ]:
#4. Average Rating by Topic :-

avg_rating_topic = df.groupby("dominant_topic")["rating"].mean()

avg_rating_topic.sort_values()

In [ ]:
#5. Create a Priority Score :-

topic_volume = df["dominant_topic"].value_counts()
topic_priority = (
    (5 - avg_rating_topic) * topic_volume
).sort_values(ascending=False)

topic_priority

In [ ]:
#6. Priority Visualization :-

plt.figure(figsize=(7,4))
sns.barplot(
    x=topic_priority.values,
    y=topic_priority.index
)
plt.title("Business Priority Score by Topic")
plt.xlabel("Priority Score")
plt.ylabel("Topic")
plt.show()

### ***-: SENTIMENT & THEME IMPACT SUMMARY :-***

**Sentiment & Theme Impact Analysis**

Combining topic modeling with customer ratings revealed that negative sentiment is highly concentrated in a small number of themes. Topics related to payments, app performance, and order issues showed both high feedback volume and low average ratings.

A priority score based on issue severity and frequency was used to rank themes, providing a data-driven framework for deciding where operational improvements would deliver the highest customer impact.

### ***-: FINAL CONCLUSION & BUSINESS RECOMMENDATION :-***

- This project analyzed unstructured customer feedback using NLP techniques to identify key themes driving customer satisfaction and dissatisfaction. Through exploratory text analysis, topic modeling (LDA), and sentiment-based impact assessment, the analysis revealed that customer issues are highly concentrated around a small number of recurring themes rather than being randomly distributed.

- Payment and refund-related issues, along with app performance problems, emerged as the most critical drivers of negative sentiment. These themes showed both high feedback volume and low average ratings, indicating a strong negative impact on overall customer experience. Order and delivery-related issues were frequent but less severe, while positive feedback tended to be generic and provided limited actionable insights.

- Based on these findings, the recommended business focus should be on stabilizing payment systems, improving refund turnaround times, and enhancing app performance through better monitoring and quality assurance. Operational improvements in order communication and customer support workflows can further reduce friction points.

- By prioritizing fixes for the top issue themes and continuously monitoring topic-level sentiment trends, organizations can achieve significant improvements in customer satisfaction with relatively targeted effort. This approach demonstrates how NLP-driven analytics can convert unstructured text data into clear, data-backed business decisions.